# Reproduce SMITH Figure 4c-h

This notebook regenerates manuscript-matched data panels from real H5AD inputs; it does not read the packaged reference-output tables. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/ribomap_section/03_SMITH_RIBOMap_Transfer_source.ipynb).

## Configure real inputs and fresh outputs

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'ribomap'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 1))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Verify input files

In [ ]:
inputs = ['ribomap_transfer/ribomap/deep_brain_ribomap.h5ad', 'ribomap_transfer/ribomap/mouse_brain_starmap_rep2.h5ad', 'ribomap_transfer/ribomap/mouse_brain_ribomap_rep2.h5ad']
rows = []
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    rows.append({"file": relative, "bytes": path.stat().st_size, "sha256": sha256_file(path)})
display(pd.DataFrame(rows))


## Run the workflow for Figure 4c-h

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/ribomap_transfer/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--methods', 'SMITH', '--panel-sizes', '32,64,128', '--training-seeds', '1,2', '--evaluation-seeds', '1,2,3', '--max-cells', '3000'] + ["--force"]
display_command = ["python", 'reproducibility/workflows/ribomap_transfer/run_tutorial.py', "--data-root", "data/tutorials", "--output-dir", "outputs/tutorials/ribomap", "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--methods', 'SMITH', '--panel-sizes', '32,64,128', '--training-seeds', '1,2', '--evaluation-seeds', '1,2,3', '--max-cells', '3000'] + ["--force"]
print(" ".join(display_command))
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
print("Generated", manifest["manuscript_figure"], "data from fresh workflow outputs.")


## Inspect newly generated figure data

In [ ]:
for relative in ['figure_data/figure4_c_f_values.tsv', 'figure_data/figure4_g_jaccard.tsv', 'figure_data/figure4_h_ribomap_bias.tsv']:
    path = CASE_OUTPUT / relative
    print(relative)
    display(pd.read_csv(path, sep="\t").head(20))


## Draw separate manuscript panels

Every panel below has its own canvas and manuscript-matched aspect ratio. The quick hosted run uses only the methods/repeats executed above; use the full command to regenerate the complete multi-method comparison.

In [ ]:
figure_dir = CASE_OUTPUT / "figures"
plot_command = [sys.executable, str(ROOT / 'reproducibility/workflows/ribomap_transfer/plot_figure4.py'), '--metrics', str(CASE_OUTPUT / 'figure_data/figure4_c_f_values.tsv'), '--overlap', str(CASE_OUTPUT / 'figure_data/figure4_g_jaccard.tsv'), '--bias', str(CASE_OUTPUT / 'figure_data/figure4_h_ribomap_bias.tsv'), "--output-dir", str(figure_dir)]
subprocess.run(plot_command, cwd=ROOT, check=True)
for heading, relative, width in [('Figure 4c - Deep-RIBOmap cell-type transfer', 'figures/figure4_c.png', 430), ('Figure 4d - Deep-RIBOmap region transfer', 'figures/figure4_d.png', 430), ('Figure 4e - STARmap cell-type transfer', 'figures/figure4_e.png', 430), ('Figure 4f - STARmap region transfer', 'figures/figure4_f.png', 430), ('Figure 4g - same- versus cross-modality overlap', 'figures/figure4_g.png', 430), ('Figure 4h - RIBOMap expression bias', 'figures/figure4_h.png', 430), ('Shared method legend', 'figures/figure4_method_legend.png', 900)]:
    display(Markdown(f"### {heading}"))
    display(Image(filename=str(CASE_OUTPUT / relative), width=width))
print("Each panel is also exported independently as editable PDF/SVG and 600-dpi TIFF under", figure_dir)


## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--methods SMITH,PERSIST-class,PERSIST,ActiveSVM,scGIST,scGeneFit,Spapros --baseline-root external/SMITH_baselines/GPS_tools-main/baselines --baseline-python PERSIST=/opt/envs/persist/bin/python --baseline-python PERSIST-class=/opt/envs/persist/bin/python --baseline-python scGIST=/opt/envs/scgist/bin/python --panel-sizes 32,64,128 --training-seeds 1,2,3,4,5 --evaluation-seeds 1,2,3,4,5 --epochs 200
```

The workflow reproduces the quantitative logic and layout of Figure 4c-h from newly selected Deep-RIBOmap and STARmap panels. Figure 4i is deliberately omitted unless a versioned Reactome/GO snapshot is supplied. Figure 4j-n additionally require the manuscript clean-fusion aligned H5AD and are not replaced with unrelated summaries.